# 06.4 — Interactive UMAP Pipeline (Experimentation Notebook)

Runs the reusable UMAP visualization pipeline and generates interactive HTML plots.
This notebook is for experimentation — modify freely without affecting 06.1/06.2/06.3.

**Environment:** `analysis` conda env (scanpy 1.12, umap-learn 0.5.11, plotly 6.7)

**Pipeline module:** `speciesOT/visualization/umap_pipeline.py`
- `compute_and_save_umap()` — PCA(50) → neighbors(15) → UMAP(0.3, seed=42) → .npz
- `generate_interactive_plots()` — .npz → interactive HTML files

In [1]:
import sys
sys.path.insert(0, "/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT")

import numpy as np
import pandas as pd
from pathlib import Path
from visualization.umap_pipeline import compute_and_save_umap, generate_interactive_plots, _load_npz

RESULTS = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/results/speciesot_cd8")
ANALYSIS_DIR = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis")

DATA_PATH = Path("/n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout_v07.h5ad")
if not DATA_PATH.exists():
    DATA_PATH = RESULTS / "data.h5ad"

PREDICTIONS = {
    "IMPACT-OR": RESULTS / "impact_or" / "evals_ood_data_space" / "imputed.h5ad",
    "CellOT": RESULTS / "cellot" / "evals_ood_data_space" / "imputed.h5ad",
}

NPZ_PATH = ANALYSIS_DIR / "umap_reference_cd8.npz"
OUTPUT_DIR = ANALYSIS_DIR / "interactive_plots"

print(f"Data: {DATA_PATH}")
print(f"NPZ: {NPZ_PATH}")
print("Setup complete.")

Data: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/cellot/cellot_gpu/datasets/speciesot-human-mouse/cd8_holdout_v07.h5ad
NPZ: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/umap_reference_cd8.npz
Setup complete.


## Step 1: Compute Reference UMAP

Only needs to run once (or when preprocessing changes). If the .npz already
exists, skip to Step 2.

In [2]:
if not NPZ_PATH.exists():
    compute_and_save_umap(
        data_path=DATA_PATH,
        output_path=NPZ_PATH,
        predictions=PREDICTIONS,
        n_comps=50,
        n_neighbors=15,
        min_dist=0.3,
        random_state=42,
    )
else:
    print(f"NPZ already exists: {NPZ_PATH} ({NPZ_PATH.stat().st_size / 1e6:.1f} MB)")
    print("Delete it and re-run this cell to recompute.")

NPZ already exists: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/umap_reference_cd8.npz (3.8 MB)
Delete it and re-run this cell to recompute.


## Step 2: Generate Standard Interactive Plots

Produces all 5 HTML files: cell types, species overlay, marker genes, model predictions.

In [3]:
generate_interactive_plots(
    npz_path=NPZ_PATH,
    output_dir=OUTPUT_DIR,
    holdout_ct_id="CL:0000625",
)

Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/interactive_all_celltypes.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/interactive_species_overlay.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/interactive_marker_genes.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/interactive_impact_or.html
Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/interactive_cellot.html

All interactive plots saved to /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/


## Step 3: Pairwise Cell Type Comparison

Select any 2 cell types to see them highlighted against a gray background.
Useful for checking overlap between:
- CD8 and thymocyte (ontology leakage concern)
- Non-classical and generic monocyte
- Any other pair of interest

In [4]:
import plotly.graph_objects as go

d = _load_npz(NPZ_PATH)
umap = d["ref_umap"]
obs = d["obs"]
ct_col = "cell_type" if "cell_type" in obs.columns else "cell_type_ontology_term_id"
all_types = sorted(obs[ct_col].unique())

print(f"Available cell types ({len(all_types)}):")
for i, ct in enumerate(all_types):
    n = (obs[ct_col] == ct).sum()
    print(f"  {i:2d}. {ct} (n={n})")

Available cell types (30):
   0. B cell (n=406)
   1. CD4-positive, alpha-beta T cell (n=190)
   2. CD8-positive, alpha-beta T cell (n=390)
   3. T cell (n=204)
   4. basophil (n=286)
   5. bronchial smooth muscle cell (n=440)
   6. classical monocyte (n=182)
   7. endothelial cell (n=406)
   8. erythrocyte (n=2)
   9. fibroblast (n=76)
  10. fibroblast of cardiac tissue (n=484)
  11. hematopoietic precursor cell (n=1584)
  12. hematopoietic stem cell (n=1442)
  13. intermediate monocyte (n=1008)
  14. large intestine goblet cell (n=760)
  15. macrophage (n=204)
  16. monocyte (n=160)
  17. myeloid dendritic cell (n=66)
  18. natural killer cell (n=912)
  19. neutrophil (n=28)
  20. non-classical monocyte (n=852)
  21. pericyte (n=134)
  22. plasma cell (n=410)
  23. plasmacytoid dendritic cell (n=54)
  24. pulmonary alveolar type 1 cell (n=8)
  25. pulmonary alveolar type 2 cell (n=584)
  26. regular atrial cardiac myocyte (n=26)
  27. smooth muscle cell (n=54)
  28. thymocyte (n=910)

In [5]:
def build_pairwise_comparison(d, type1, type2, output_path=None):
    """Build interactive plot comparing two cell types against gray background."""
    umap = d["ref_umap"]
    obs = d["obs"]
    ct_col = "cell_type" if "cell_type" in obs.columns else "cell_type_ontology_term_id"
    
    mask1 = obs[ct_col].values == type1
    mask2 = obs[ct_col].values == type2
    bg = ~(mask1 | mask2)
    
    fig = go.Figure()
    
    def _hover(mask, label):
        return [
            f"<b>{label}</b><br>Condition: {obs.get('condition', pd.Series(['N/A']*len(obs))).values[i]}"
            f"<br>Tissue: {obs.get('tissue', pd.Series(['N/A']*len(obs))).values[i]}"
            f"<br>Donor: {obs.get('donor_id', pd.Series(['N/A']*len(obs))).values[i]}"
            f"<br>UMAP: ({umap[i,0]:.2f}, {umap[i,1]:.2f})"
            for i in np.where(mask)[0]
        ]
    
    fig.add_trace(go.Scattergl(
        x=umap[bg, 0], y=umap[bg, 1],
        mode="markers", marker=dict(size=3, color="#d0d0d0", opacity=0.2),
        name=f"Other (n={bg.sum()})", hoverinfo="skip",
    ))
    
    fig.add_trace(go.Scattergl(
        x=umap[mask1, 0], y=umap[mask1, 1],
        mode="markers", marker=dict(size=6, color="#E24A33", opacity=0.8),
        name=f"{type1} (n={mask1.sum()})",
        hovertext=_hover(mask1, type1), hoverinfo="text",
    ))
    
    fig.add_trace(go.Scattergl(
        x=umap[mask2, 0], y=umap[mask2, 1],
        mode="markers", marker=dict(size=6, color="#348ABD", opacity=0.8),
        name=f"{type2} (n={mask2.sum()})",
        hovertext=_hover(mask2, type2), hoverinfo="text",
    ))
    
    fig.update_layout(
        title=f"Pairwise: {type1} vs {type2}",
        xaxis_title="UMAP 1", yaxis_title="UMAP 2",
        plot_bgcolor="white", width=1100, height=900,
        legend=dict(font=dict(size=12)),
        xaxis=dict(showgrid=False, zeroline=False),
        yaxis=dict(showgrid=False, zeroline=False),
    )
    
    if output_path:
        fig.write_html(str(output_path), auto_open=False)
        print(f"Saved: {output_path}")
    
    fig.show()
    return fig


build_pairwise_comparison(
    d,
    "CD8-positive, alpha-beta T cell",
    "thymocyte",
    output_path=OUTPUT_DIR / "pairwise_cd8_vs_thymocyte.html",
)

Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/pairwise_cd8_vs_thymocyte.html


In [6]:
build_pairwise_comparison(
    d,
    "CD8-positive, alpha-beta T cell",
    "T cell",
    output_path=OUTPUT_DIR / "pairwise_cd8_vs_tcell.html",
)

Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/pairwise_cd8_vs_tcell.html


In [7]:
build_pairwise_comparison(
    d,
    "non-classical monocyte",
    "monocyte",
    output_path=OUTPUT_DIR / "pairwise_nonclassical_vs_generic_mono.html",
)

Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/pairwise_nonclassical_vs_generic_mono.html


In [8]:
build_pairwise_comparison(
    d,
    "CD8-positive, alpha-beta T cell",
    "pulmonary alveolar type 2 cell",
    output_path=OUTPUT_DIR / "pairwise_cd8_vs_pulmonary_alveolar.html",
)

Saved: /n/holylabs/mooney_lab/Lab/junyizhou/speciesOT/speciesOT/baseline/analysis/interactive_plots/pairwise_cd8_vs_pulmonary_alveolar.html


## Custom Comparisons

Change the two cell type names below to compare any pair:

In [9]:
TYPE_A = "CD4-positive, alpha-beta T cell"
TYPE_B = "CD8-positive, alpha-beta T cell"

build_pairwise_comparison(d, TYPE_A, TYPE_B)

## Notes

- **06.1**: Static matplotlib UMAP (original, speciesOT_env)
- **06.2**: Calls the pipeline to compute + generate all interactive HTMLs
- **06.3**: Static scanpy sc.pl.umap panels (speciesOT_env)
- **06.4** (this notebook): Experimentation — pipeline + pairwise comparison + custom plots

When preprocessing changes, delete the .npz file and re-run this notebook.